In [ ]:
# # import os
# import random
# import datetime
# import hashlib
# import json
# from getpass import getpass
# import mysql.connector
# from mysql.connector import Error

# # Database configuration
# DB_CONFIG = {
#     'host': 'localhost',
#     'user': 'bank_user',
#     'password': 'bank_password',
#     'database': 'bank_system'
# }

# class Account:
#     def _init_(self, acc_no, name, pin, balance=0.0, acc_type="Savings", created_at=None):
#         self.acc_no = acc_no
#         self.name = name
#         self.pin = pin
#         self.balance = balance
#         self.acc_type = acc_type
#         self.created_at = created_at or datetime.datetime.now().strftime("%d/%m/%Y %H:%M %P")

#     def to_dict(self):
#         return {
#             "acc_no": self.acc_no,
#             "name": self.name,
#             "pin": self.pin,
#             "balance": self.balance,
#             "acc_type": self.acc_type,
#             "created_at": self.created_at
#         }

#     @classmethod
#     def from_dict(cls, data):
#         return cls(
#             data["acc_no"],
#             data["name"],
#             data["pin"],
#             data["balance"],
#             data["acc_type"],
#             data.get("created_at")
#         )

# class Transaction:
#     def _init_(self, txn_id=None, acc_no=None, amount=0.0, transaction_type="", description="", timestamp=None):
#         self.txn_id = txn_id
#         self.acc_no = acc_no
#         self.amount = amount
#         self.transaction_type = transaction_type
#         self.description = description
#         self.timestamp = timestamp or datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

#     def to_dict(self):
#         return {
#             "txn_id": self.txn_id,
#             "acc_no": self.acc_no,
#             "amount": self.amount,
#             "transaction_type": self.transaction_type,
#             "description": self.description,
#             "timestamp": self.timestamp
#         }

#     @classmethod
#     def from_dict(cls, data):
#         return cls(
#             data.get("txn_id"),
#             data["acc_no"],
#             data["amount"],
#             data["transaction_type"],
#             data["description"],
#             data.get("timestamp")
#         )

# # Database Operations
# def create_connection():
#     try:
#         connection = mysql.connector.connect(**DB_CONFIG)
#         return connection
#     except Error as e:
#         print(f"Error connecting to MySQL: {e}")
#         return None

# def initialize_database():
#     connection = create_connection()
#     if connection:
#         try:
#             cursor = connection.cursor()
            
#             # Create accounts table
#             cursor.execute("""
#                 CREATE TABLE IF NOT EXISTS accounts (
#                     acc_no VARCHAR(10) PRIMARY KEY,
#                     name VARCHAR(100) NOT NULL,
#                     pin VARCHAR(64) NOT NULL,
#                     balance DECIMAL(15, 2) NOT NULL DEFAULT 0.0,
#                     acc_type VARCHAR(20) NOT NULL,
#                     created_at DATETIME NOT NULL
#                 )
#             """)
            
#             # Create transactions table
#             cursor.execute("""
#                 CREATE TABLE IF NOT EXISTS transactions (
#                     txn_id INT AUTO_INCREMENT PRIMARY KEY,
#                     acc_no VARCHAR(10) NOT NULL,
#                     amount DECIMAL(15, 2) NOT NULL,
#                     transaction_type VARCHAR(20) NOT NULL,
#                     description VARCHAR(255),
#                     timestamp DATETIME NOT NULL,
#                     FOREIGN KEY (acc_no) REFERENCES accounts(acc_no)
#                 )
#             """)
            
#             connection.commit()
#             print("Database initialized successfully")
#         except Error as e:
#             print(f"Error initializing database: {e}")
#         finally:
#             cursor.close()
#             connection.close()

# def load_accounts():
#     accounts = {}
#     connection = create_connection()
#     if connection:
#         try:
#             cursor = connection.cursor(dictionary=True)
#             cursor.execute("SELECT * FROM accounts")
#             rows = cursor.fetchall()
            
#             for row in rows:
#                 account = Account(
#                     row['acc_no'],
#                     row['name'],
#                     row['pin'],
#                     float(row['balance']),
#                     row['acc_type'],
#                     row['created_at'].strftime("%d/%m/%Y %H:%M %P") if row['created_at'] else None
#                 )
#                 accounts[row['acc_no']] = account
#         except Error as e:
#             print(f"Error loading accounts: {e}")
#         finally:
#             cursor.close()
#             connection.close()
#     return accounts

# def save_account(account):
#     connection = create_connection()
#     if connection:
#         try:
#             cursor = connection.cursor()
            
#             # Check if account exists
#             cursor.execute("SELECT 1 FROM accounts WHERE acc_no = %s", (account.acc_no,))
#             exists = cursor.fetchone()
            
#             if exists:
#                 # Update existing account
#                 query = """
#                     UPDATE accounts 
#                     SET name = %s, pin = %s, balance = %s, acc_type = %s 
#                     WHERE acc_no = %s
#                 """
#                 cursor.execute(query, (
#                     account.name,
#                     account.pin,
#                     account.balance,
#                     account.acc_type,
#                     account.acc_no
#                 ))
#             else:
#                 # Insert new account
#                 query = """
#                     INSERT INTO accounts (acc_no, name, pin, balance, acc_type, created_at)
#                     VALUES (%s, %s, %s, %s, %s, %s)
#                 """
#                 cursor.execute(query, (
#                     account.acc_no,
#                     account.name,
#                     account.pin,
#                     account.balance,
#                     account.acc_type,
#                     account.created_at
#                 ))
            
#             connection.commit()
#         except Error as e:
#             print(f"Error saving account: {e}")
#             connection.rollback()
#         finally:
#             cursor.close()
#             connection.close()

# def delete_account(acc_no):
#     connection = create_connection()
#     if connection:
#         try:
#             cursor = connection.cursor()
            
#             # First delete related transactions
#             cursor.execute("DELETE FROM transactions WHERE acc_no = %s", (acc_no,))
            
#             # Then delete the account
#             cursor.execute("DELETE FROM accounts WHERE acc_no = %s", (acc_no,))
            
#             connection.commit()
#             return cursor.rowcount > 0
#         except Error as e:
#             print(f"Error deleting account: {e}")
#             connection.rollback()
#             return False
#         finally:
#             cursor.close()
#             connection.close()

# def load_transactions(acc_no=None):
#     transactions = []
#     connection = create_connection()
#     if connection:
#         try:
#             cursor = connection.cursor(dictionary=True)
            
#             if acc_no:
#                 cursor.execute("SELECT * FROM transactions WHERE acc_no = %s ORDER BY timestamp DESC", (acc_no,))
#             else:
#                 cursor.execute("SELECT * FROM transactions ORDER BY timestamp DESC")
            
#             rows = cursor.fetchall()
            
#             for row in rows:
#                 transaction = Transaction(
#                     row['txn_id'],
#                     row['acc_no'],
#                     float(row['amount']),
#                     row['transaction_type'],
#                     row['description'],
#                     row['timestamp'].strftime("%Y-%m-%d %H:%M:%S") if row['timestamp'] else None
#                 )
#                 transactions.append(transaction)
#         except Error as e:
#             print(f"Error loading transactions: {e}")
#         finally:
#             cursor.close()
#             connection.close()
#     return transactions

# def save_transaction(transaction):
#     connection = create_connection()
#     if connection:
#         try:
#             cursor = connection.cursor()
            
#             query = """
#                 INSERT INTO transactions (acc_no, amount, transaction_type, description, timestamp)
#                 VALUES (%s, %s, %s, %s, %s)
#             """
#             cursor.execute(query, (
#                 transaction.acc_no,
#                 transaction.amount,
#                 transaction.transaction_type,
#                 transaction.description,
#                 transaction.timestamp
#             ))
            
#             connection.commit()
#             return cursor.lastrowid
#         except Error as e:
#             print(f"Error saving transaction: {e}")
#             connection.rollback()
#             return None
#         finally:
#             cursor.close()
#             connection.close()

# # Helper Functions
# def generate_account_number():
#     while True:
#         acc_no = ''.join([str(random.randint(0, 9)) for _ in range(10)])
        
#         # Check if account number already exists
#         connection = create_connection()
#         if connection:
#             try:
#                 cursor = connection.cursor()
#                 cursor.execute("SELECT 1 FROM accounts WHERE acc_no = %s", (acc_no,))
#                 if not cursor.fetchone():
#                     return acc_no
#             except Error as e:
#                 print(f"Error checking account number: {e}")
#                 return None
#             finally:
#                 cursor.close()
#                 connection.close()

# def hash_pin(pin):
#     return hashlib.sha256(pin.encode()).hexdigest()

# def verify_pin(hashed_pin, user_pin):
#     return hashed_pin == hash_pin(user_pin)

# # Bank Operations
# def create_account(accounts):
#     print("\n=======CREATE NEW ACCOUNT========")
    
#     name = input("Enter your name: ")
    
#     while True:
#         pin = getpass("Enter 4-Digit PIN: ")
#         if len(pin) == 4 and pin.isdigit():
#             confirm_pin = getpass("Confirm PIN: ")
            
#             if pin == confirm_pin:
#                 break
#             else:
#                 print("PINs do not match. Try Again")
#         else:
#             print("PIN must be 4 digits")
    
#     acc_type = input("Enter account type (Savings/Current): ").capitalize()
#     if acc_type not in ["Savings", "Current"]:
#         acc_type = "Savings"
    
#     initial_deposit = float(input("Enter initial deposit amount: "))
    
#     acc_no = generate_account_number()
#     if not acc_no:
#         print("Error generating account number")
#         return
    
#     hashed_pin = hash_pin(pin)
#     account = Account(acc_no, name, hashed_pin, initial_deposit, acc_type)
    
#     # Save account to database
#     save_account(account)
    
#     # Record transaction
#     save_transaction(Transaction(acc_no, initial_deposit, "Deposit", "Initial Deposit"))
    
#     print(f"\nAccount created successfully!\nAccount Number: {acc_no}")

# def login(accounts):
#     print("\n=======Login=======")
#     acc_no = input("Enter Account No.: ")
#     pin = getpass("Enter PIN: ")
    
#     connection = create_connection()
#     if connection:
#         try:
#             cursor = connection.cursor(dictionary=True)
#             cursor.execute("SELECT * FROM accounts WHERE acc_no = %s", (acc_no,))
#             row = cursor.fetchone()
            
#             if row:
#                 if verify_pin(row['pin'], pin):
#                     account = Account(
#                         row['acc_no'],
#                         row['name'],
#                         row['pin'],
#                         float(row['balance']),
#                         row['acc_type'],
#                         row['created_at'].strftime("%d/%m/%Y %H:%M %P") if row['created_at'] else None
#                     )
#                     return account
#                 else:
#                     print("Invalid PIN")
#             else:
#                 print("Account not found")
#         except Error as e:
#             print(f"Error during login: {e}")
#         finally:
#             cursor.close()
#             connection.close()
    
#     return None

# def deposit(account, accounts):
#     print("\n======DEPOSIT========")
#     amount = float(input("Enter amount: "))
    
#     if amount > 0:
#         # Update account balance in database
#         connection = create_connection()
#         if connection:
#             try:
#                 cursor = connection.cursor()
                
#                 # Update balance
#                 cursor.execute(
#                     "UPDATE accounts SET balance = balance + %s WHERE acc_no = %s",
#                     (amount, account.acc_no)
#                 )
                
#                 # Record transaction
#                 save_transaction(Transaction(account.acc_no, amount, "Deposit"))
                
#                 connection.commit()
                
#                 # Update local account object
#                 account.balance += amount
#                 print(f"Deposit Successful\nNew balance: {account.balance}")
#             except Error as e:
#                 print(f"Error during deposit: {e}")
#                 connection.rollback()
#             finally:
#                 cursor.close()
#                 connection.close()
#     else:
#         print("Invalid Amount")

# def withdraw(account, accounts):
#     print("\n======WITHDRAW========")
#     amount = float(input("Enter amount: "))
    
#     if amount > 0:
#         if account.balance >= amount:
#             # Update account balance in database
#             connection = create_connection()
#             if connection:
#                 try:
#                     cursor = connection.cursor()
                    
#                     # Update balance
#                     cursor.execute(
#                         "UPDATE accounts SET balance = balance - %s WHERE acc_no = %s",
#                         (amount, account.acc_no)
#                     )
                    
#                     # Record transaction
#                     save_transaction(Transaction(account.acc_no, amount, "Withdrawal"))
                    
#                     connection.commit()
                    
#                     # Update local account object
#                     account.balance -= amount
#                     print(f"Withdrawal Successful\nNew balance: {account.balance}")
#                 except Error as e:
#                     print(f"Error during withdrawal: {e}")
#                     connection.rollback()
#                 finally:
#                     cursor.close()
#                     connection.close()
#         else:
#             print("Insufficient Balance")
#     else:
#         print("Invalid Amount")

# def transfer(account, accounts):
#     print("\n======TRANSFER========")
#     target_acc_no = input("Enter beneficiary account number: ")
#     amount = float(input("Enter amount: "))
    
#     if target_acc_no == account.acc_no:
#         print("Cannot transfer to the same account")
#         return
    
#     if amount <= 0:
#         print("Invalid Amount")
#         return
    
#     connection = create_connection()
#     if not connection:
#         print("Database connection error")
#         return
    
#     try:
#         cursor = connection.cursor(dictionary=True)
        
#         # Check if target account exists
#         cursor.execute("SELECT 1 FROM accounts WHERE acc_no = %s", (target_acc_no,))
#         if not cursor.fetchone():
#             print("Beneficiary Account Not Found")
#             return
        
#         # Check if sender has sufficient balance
#         cursor.execute("SELECT balance FROM accounts WHERE acc_no = %s", (account.acc_no,))
#         sender_balance = float(cursor.fetchone()['balance'])
        
#         if sender_balance < amount:
#             print("Insufficient Balance")
#             return
        
#         # Start transaction
#         connection.start_transaction()
        
#         # Deduct from sender
#         cursor.execute(
#             "UPDATE accounts SET balance = balance - %s WHERE acc_no = %s",
#             (amount, account.acc_no)
#         )
        
#         # Add to receiver
#         cursor.execute(
#             "UPDATE accounts SET balance = balance + %s WHERE acc_no = %s",
#             (amount, target_acc_no)
#         )
        
#         # Record transactions
#         save_transaction(Transaction(
#             account.acc_no,
#             amount,
#             "Transfer",
#             f"To {target_acc_no}"
#         ))
        
#         save_transaction(Transaction(
#             target_acc_no,
#             amount,
#             "Transfer",
#             f"From {account.acc_no}"
#         ))
        
#         connection.commit()
        
#         # Update local account object
#         account.balance -= amount
#         print(f"Transfer Successful\nNew balance: {account.balance}")
        
#     except Error as e:
#         print(f"Error during transfer: {e}")
#         connection.rollback()
#     finally:
#         cursor.close()
#         connection.close()

# def view_balance(account):
#     print("\n=======Account INFO========")
#     print(f"Account Number: {account.acc_no}")
#     print(f"Account Holder: {account.name}")
#     print(f"Account Type: {account.acc_type}")
#     print(f"Account Balance: {account.balance}")
#     print(f"Account Created: {account.created_at}")

# def view_transactions(account):
#     print("\n=======TRANSACTION HISTORY=========")
#     transactions = load_transactions(account.acc_no)
    
#     if not transactions:
#         print("No Transaction Found")
#         return
    
#     print(f"Transactions for Account: {account.acc_no}")
#     print("-" * 50)
#     print(f"{'Date':<20} {'Type':<15} {'Amount':<15} {'Description'}")
#     print("-" * 50)
    
#     for txn in transactions:
#         print(f"{txn.timestamp:<20} {txn.transaction_type:<15} {txn.amount:<15.2f} {txn.description}")

# def update_account(account):
#     print("\n=======UPDATE ACCOUNT========")
#     print("1. Change Name")
#     print("2. Change PIN")
#     print("3. Back")
    
#     choice = input("Enter your choice: ")
    
#     if choice == "1":
#         new_name = input("Enter new name: ")
#         if new_name:
#             connection = create_connection()
#             if connection:
#                 try:
#                     cursor = connection.cursor()
#                     cursor.execute(
#                         "UPDATE accounts SET name = %s WHERE acc_no = %s",
#                         (new_name, account.acc_no)
#                     )
#                     connection.commit()
#                     account.name = new_name
#                     print("Name updated successfully")
#                 except Error as e:
#                     print(f"Error updating name: {e}")
#                     connection.rollback()
#                 finally:
#                     cursor.close()
#                     connection.close()
    
#     elif choice == "2":
#         while True:
#             current_pin = getpass("Enter current PIN: ")
#             if verify_pin(account.pin, current_pin):
#                 new_pin = getpass("Enter new 4-Digit PIN: ")
#                 if len(new_pin) == 4 and new_pin.isdigit():
#                     confirm_pin = getpass("Confirm new PIN: ")
                    
#                     if new_pin == confirm_pin:
#                         hashed_pin = hash_pin(new_pin)
#                         connection = create_connection()
#                         if connection:
#                             try:
#                                 cursor = connection.cursor()
#                                 cursor.execute(
#                                     "UPDATE accounts SET pin = %s WHERE acc_no = %s",
#                                     (hashed_pin, account.acc_no)
#                                 )
#                                 connection.commit()
#                                 account.pin = hashed_pin
#                                 print("PIN changed successfully")
#                                 break
#                             except Error as e:
#                                 print(f"Error changing PIN: {e}")
#                                 connection.rollback()
#                             finally:
#                                 cursor.close()
#                                 connection.close()
#                     else:
#                         print("PINs do not match")
#                 else:
#                     print("PIN must be 4 digits")
#             else:
#                 print("Incorrect current PIN")
#                 break

# def delete_account_menu(account, accounts):
#     print("\n=======DELETE ACCOUNT========")
#     confirm = input("Are you sure you want to delete your account? (yes/no): ").lower()
    
#     if confirm == "yes":
#         pin = getpass("Enter your PIN to confirm: ")
        
#         if verify_pin(account.pin, pin):
#             if delete_account(account.acc_no):
#                 print("Account deleted successfully")
#                 return True  # Signal to logout
#             else:
#                 print("Failed to delete account")
#         else:
#             print("Incorrect PIN")
    
#     return False

# def main():
#     # Initialize database on first run
#     initialize_database()
    
#     current_account = None
    
#     while True:
#         if current_account is None:
#             print("\n=======WELCOME TO BOI=======")
#             print("1. Create New Account")
#             print("2. Login")
#             print("3. Exit")
            
#             choice = input("Enter your choice: ")
        
#             if choice == "1":
#                 create_account(load_accounts())
#             elif choice == "2":
#                 current_account = login(load_accounts())
#             elif choice == "3":
#                 break
#             else:
#                 print("Invalid choice. Please try again.")
        
#         else:
#             print(f"\n=======WELCOME {current_account.name}=======")
#             print("1. Deposit")
#             print("2. Withdraw")
#             print("3. Transfer")
#             print("4. View Balance")
#             print("5. View Transactions")
#             print("6. Update Account")
#             print("7. Delete Account")
#             print("8. Logout")
    
#             choice = input("Enter your choice: ")

#             if choice == "1":
#                 deposit(current_account, load_accounts())
#             elif choice == "2":
#                 withdraw(current_account, load_accounts())
#             elif choice == "3":
#                 transfer(current_account, load_accounts())
#             elif choice == "4":
#                 view_balance(current_account)
#             elif choice == "5":
#                 view_transactions(current_account)
#             elif choice == "6":
#                 update_account(current_account)
#             elif choice == "7":
#                 if delete_account_menu(current_account, load_accounts()):
#                     current_account = None
#             elif choice == "8":
#                 current_account = None
#                 print("Logged out successfully.")
#             else:
#                 print("Invalid choice. Please try again.")

# if _name_ == "_main_":
#     main()